# ERP Workflow

This notebook explains how ERP receives client orders and dispatches production plans.

## ERP scope
- Receive client orders via TCP 6666
- Persist in erp_db
- Build MPS and purchasing plan
- Dispatch production lines to MES using database inserts

## ERP Dashboard and Full Component Run

Use the ERP launcher to start components with structured logs:
- `python ERP/run_erp_programs.py --mode all`
- `python ERP/run_erp_programs.py --mode demo` (full stack + demo orders for testing)

Dashboard entrypoint (single component):
- `streamlit run ERP/erp_gui.py`

`demo` mode is intended for quick validation of end-to-end behavior without manual order entry.

### ERP launcher
This code block starts the full ERP program stack and opens the graphical ERP dashboard in your browser. The terminal only launches the services; the interface itself is a web GUI.

## Implemented Features and Specification Mapping

### Requirement 3.1 — Client Orders
- ERP accepts client orders over **TCP port 6666** in JSON format.
- Payload is parsed into client metadata (`name`, `NIF`, `OrderID`) and line items (`type`, `quantity`, `DDate`, `Penalty`).
- Orders are persisted into `erp_db` with normalized structure:
  - `Clients`
  - `Orders`
  - `Order_Items`

### Requirement 3.2 — Plans (implemented baseline)
- `erp_mps.py` provides simulation-day progression (`1 day = 60 seconds`) and generates a planning horizon view.
- Purchasing recommendations are derived from BOM and supplier constraints for Wood/Metal.
- Dispatcher sends executable production lines to MES through database writes.

## Component Structure
- `ERP/erp_server.py`: ingestion boundary for client JSON over TCP 6666 and persistence to `erp_db`.
- `ERP/erp_mps.py`: planning and purchasing logic tied to simulation day.
- `ERP/erp_dispatcher.py`: operational dispatcher that transfers due production lines to MES queue tables.
- `ERP/erp_gui.py`: Streamlit interface for:
  - manual order entry form
  - MPS visualization
  - JSON upload parsing and submission workflow

## ERP Execution Flow
1. ERP server listens on TCP 6666.
2. Incoming JSON is parsed and validated.
3. ERP stores order data in `erp_db`.
4. Dispatcher reads pending schedules and posts production lines to MES database queue.
5. ERP dashboard presents current orders, plan status, and MES feedback visibility.

### ERP launcher
This code block starts the full ERP stack from the notebook so you can launch the server, dispatcher, and dashboard together.

In [2]:
from pathlib import Path
import subprocess, sys

root = Path.cwd()
erp_dir = root / 'ERP'

def run_erp_mode(mode='all'):
    cmd = [sys.executable, str(erp_dir / 'run_erp_programs.py'), '--mode', mode]
    print('Running:', ' '.join(cmd))
    print('Modes: server, dispatcher, dashboard, all, demo')
    print('The ERP dashboard will open in your browser on port 8501.')
    return subprocess.Popen(cmd)

# Run the full ERP component from this notebook:
# p_all = run_erp_mode('all')
# p_demo = run_erp_mode('demo')

### ERP order sender
This code block prepares a sample client order payload and sends it to the ERP server so you can exercise the live order intake path.

In [ ]:
import socket, json

sample_payload = {
    'name': 'Client Demo',
    'NIF': 123456789,
    'OrderID': 1001,
    'orders': [
        {'type': 'RWM', 'quantity': 3, 'DDate': 4, 'Penalty': 2.5},
        {'type': 'SMM', 'quantity': 2, 'DDate': 6, 'Penalty': 3.0}
    ]